In [1]:
import os
from io import BytesIO
from skimage import io
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import folium
import urllib.request
import urllib.parse
import mapbox_vector_tile
import xml.etree.ElementTree as xmlet
import lxml.etree as xmltree
from PIL import Image as plimg
from PIL import ImageDraw
import numpy as np
import pandas as pd
from owslib.wms import WebMapService
from IPython.display import Image, display
import geopandas as gpd
from shapely.geometry import box
import urllib.request
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import show
import fiona
from datetime import datetime, timedelta
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
%matplotlib inline

In [2]:
lat, lon = -15, -75  # West coast South America
def latlon_to_pixel(lat, lon, width=800, height=400):
    minx, miny, maxx, maxy = -180, -60, 180, 60

    x = int((lon - minx) / (maxx - minx) * width)
    y = int((maxy - lat) / (maxy - miny) * height)

    return x, y

def latlon_to_pixel(lat, lon, width=800, height=400):
    minx, miny, maxx, maxy = -180, -60, 180, 60

    x = int((lon - minx) / (maxx - minx) * width)
    y = int((maxy - lat) / (maxy - miny) * height)

    return x, y

def get_redness(image_path, lat, lon):
    img = Image.open(image_path).convert("RGB")
    x, y = latlon_to_pixel(lat, lon)

    r, g, b = img.getpixel((x, y))

    # simple “redness” metric
    redness = r - (g + b) / 2

    return redness

def show_map(year):
    display(Image.open(f"el_nino_year/{year}.png"))


In [3]:



# -----------------------------
# CONFIG
# -----------------------------
start_year = 2019
end_year = 2025
months = list(range(1, 13))

timeline = [(2019,m) for m in list(range(8,13))]+[(y, m) for y in range(start_year+1, end_year + 1) for m in months]


# -----------------------------
# MAP FUNCTION
# -----------------------------
def show_map(year, month):
    img = Image.open(f"el_nino/{year}_{month:02d}_22.png")
    display(img)


# -----------------------------
# SLIDER
# -----------------------------
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(timeline) - 1,
    step=1,
    continuous_update=True,
    layout=widgets.Layout(width="800px")
)

slider.readout = False  # removes ugly number


# -----------------------------
# OUTPUT AREA
# -----------------------------
output = widgets.Output()


# -----------------------------
# UPDATE FUNCTION
# -----------------------------
def update(change):
    i = change["new"]
    year, month = timeline[i]

    with output:
        clear_output(wait=True)
        show_map(year, month)


slider.observe(update, names="value")


# -----------------------------
# INITIAL IMAGE
# -----------------------------
with output:
    clear_output(wait=True)
    y, m = timeline[0]
    show_map(y, m)


# -----------------------------
# TICK RULER (FIXED NO CUT-OFF)
# -----------------------------
ticks = widgets.HTML()
ticks.value = """
<div style="
    width:800px;
    position:relative;
    height:60px;
    margin-top:5px;
    overflow:visible;
    padding-right:25px;
    box-sizing:border-box;
    font-size:10px;
">
"""

total = len(timeline) - 1

for i, (y, m) in enumerate(timeline):
    left = (i / total) * 100

    height = "14px" if m == 1 else "6px"

    ticks.value += f"""
    <div style="
        position:absolute;
        left:{left}%;
        top:0px;
        width:1px;
        height:{height};
        background:black;
        opacity:0.7;
    "></div>
    """

    if m == 1:
        ticks.value += f"""
        <div style="
            position:absolute;
            left:calc({left}% - 10px);
            top:18px;
            font-size:10px;
            white-space:nowrap;
        ">
            {y}
        </div>
        """

ticks.value += "</div>"


# -----------------------------
# CENTERED LAYOUT
# -----------------------------
center_layout = widgets.Layout(
    display='flex',
    flex_flow='column',
    align_items='center',
    width='100%'
)

slider_box = widgets.VBox(
    [slider],
    layout=widgets.Layout(margin="10px 0px 5px 0px", width="800px")
)

ticks_box = widgets.VBox(
    [ticks],
    layout=widgets.Layout(margin="0px 0px 25px 0px", width="800px")
)

output_box = widgets.VBox(
    [output],
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="800px")
)


# -----------------------------
# FINAL UI
# -----------------------------
ui = widgets.VBox(
    [slider_box, ticks_box, output_box],
    layout=center_layout
)

display(ui)